[2026-08-07 Fiddler](https://thefiddler.substack.com/p/how-lucky-can-a-baseball-team-get)
====================

Fiddler
-------
The probability that the A's (or the G-men) win $n$ games is ${162\choose n}/2^{162}$, so the probability
that the team with the better record wins $n$ games, where $n > 81$, is $2{162\choose n}/2^{162}$.

And the probability that both teams win 81 games is ${162\choose 81}/2^{162}$.

So, on average, the better team wins
$\left(81{162\choose81}+\sum_{n=82}^{162} 2n{162\choose n}\right)/2^{162} \approx 86.0699$.
[Simulations](20260807.go) agree, giving approximately 86.069.

In [1]:
p = (81*binomial(162,81)+sum(2*x*binomial(162,x), x, 82, 162))/2^162
p, numerical_approx(p)

(62895632625904371267814341812263929371665856338413/730750818665451459101842416358141509827966271488,
 86.0698763783376)

If 81-81 outcomes are ignored, then, on average, the better team wins
$\sum_{n=82}^{162} 2n{162\choose n}/\left(2^{162} - {162\choose81}\right) \approx 86.408$.

[Simulations](20260807.go) do not agree, giving approximately 91.15.  I don't know where my error is.

In [2]:
p = sum(2*x*binomial(162,x),x,82,162)/(2^162-binomial(162,81))
p, numerical_approx(p)

(59190816311901568187249235725009462296065267990528/685012345653071174156594193058703644697094810403,
 86.4083934946760)

Extra credit
------------
After two teams play each other, the possible distinct outcomes are

* probability of 2/32 that one team has 5 wins, one team has 0 wins, no teams are out of contention for the most wins
* probability of 10/32 that one team has 4 wins, one team has 1 win, no teams are out of contention for the most wins
* probability of 20/32 that one team has 3 wins, one team has 2 wins, no teams are out of contention for the most wins

I'll lump together all teams that are out of contention for the most wins to reduce the number of distinct outcomes,
since it doesn't matter how many wins they individually have.

After a third team plays both the first two teams, one possible outcome
is that one team has 10 wins, one team has 5 wins, one team has 0 wins, no teams are out of contention for the most
wins, which can happen 3 ways: the new team wins 10, the new team loses 10, the new team wins 5 from the winless team
and loses 5 to the undefeated team, making the probability of this outcome $2/32\cdot3/32^2 = 3/16384$.

Then, from all the 3-team outcomes, find the probabilities of the outcomes after a fourth team plays each
of the three teams.

This calculation becomes computationally unfeasible long before it gets to 30 teams, as there are 1111
distinct outcomes with 5 teams, and calculating the 10461 distinct outcomes with 6 teams takes minutes.
At least [simulations](20260807.go) indicate that the average number of wins the best team has should
be approximately 84.98.

One idea for speeding up this calculation would be to lump all teams with the same number of wins
together, though they'd have to be separated again.  I doubt it would be enough to get to 30 teams, though.

In [3]:
def add_team(dist, remaining_teams):
    new_dist = {}
    for team_wins, prob in dist.items():
        for new_prob, new_team_wins in dists_after_adding_team(0, team_wins):
            max_wins = max(new_team_wins)
            new_team_wins = list(new_team_wins)
            for i in range(len(new_team_wins)):
                if max_wins - new_team_wins[i] > 5*remaining_teams:
                    new_team_wins[i] = 0
            new_team_wins.sort(reverse=True)
            new_team_wins = tuple(new_team_wins)
            if new_team_wins in new_dist:
                new_dist[new_team_wins] += new_prob*prob
            else:
                new_dist[new_team_wins] = new_prob*prob
    return new_dist

def dists_after_adding_team(new_team_wins, other_teams):
    if len(other_teams) == 0:
        yield 1, (new_team_wins,)
        return
    for p, teams in dists_after_adding_team(new_team_wins, other_teams[1:]):
        yield p/32, teams + (other_teams[0]+5,)
    for p, teams in dists_after_adding_team(new_team_wins+1, other_teams[1:]):
        yield 5*p/32, teams + (other_teams[0]+4,)
    for p, teams in dists_after_adding_team(new_team_wins+2, other_teams[1:]):
        yield 10*p/32, teams + (other_teams[0]+3,)
    for p, teams in dists_after_adding_team(new_team_wins+3, other_teams[1:]):
        yield 10*p/32, teams + (other_teams[0]+2,)
    for p, teams in dists_after_adding_team(new_team_wins+4, other_teams[1:]):
        yield 5*p/32, teams + (other_teams[0]+1,)
    for p, teams in dists_after_adding_team(new_team_wins+5, other_teams[1:]):
        yield p/32, teams + (other_teams[0],)

In [4]:
dist = add_team({(0,):1}, 28)
print(2, dist)
for n_teams in [3..6]:
    dist = add_team(dist, 30-n_teams)
    if n_teams < 4:
        print(n_teams, dist)
    else:
        print(n_teams, len(dist))

2 {(5, 0): 1/16, (4, 1): 5/16, (3, 2): 5/8}
3 {(10, 5, 0): 3/16384, (10, 4, 1): 15/16384, (10, 3, 2): 15/8192, (9, 5, 1): 45/8192, (9, 4, 2): 225/16384, (9, 3, 3): 75/8192, (9, 6, 0): 15/16384, (8, 5, 2): 705/16384, (8, 4, 3): 75/1024, (8, 6, 1): 225/16384, (8, 7, 0): 15/8192, (7, 5, 3): 645/4096, (7, 4, 4): 825/8192, (7, 6, 2): 75/1024, (7, 7, 1): 75/8192, (6, 5, 4): 1335/4096, (6, 6, 3): 825/8192, (5, 5, 5): 563/8192}
4 131
5 1111
6 10461
